In this notebook, let's do our initial data exploration. Some things I think we should include:
- Stats + Distributions
- Plots (bar charts + heatmaps)
- Correlation (Chi-squared)

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt


# loading data and labeling columns

columns = ['target', 'cap-shape', 'cap-surface', 'cap-color', 'bruises',
           'odor', 'gill-attachment', 'gill-spacing', 'gill-size',
           'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
           'stalk-surface-below-ring', 'stalk-color-above-ring',
           'stalk-color-below-ring', 'veil-type', 'veil-color',
           'ring-number', 'ring-type', 'spore-print-color',
           'population', 'habitat']

df = pd.read_csv('../data/agaricus-lepiota.data', header = None, names = columns)

df.head()

In [ ]:
# target variable EDA

print(df['target'].value_counts())

df['target'].value_counts().plot(kind = 'bar')
plt.title('Distribution of Edible vs Poisonous Mushrooms')
plt.xticks(ticks=[0, 1], labels=['Edible', 'Poisonous'], rotation=0)
plt.ylabel('Count')
plt.savefig('../results/target_distribution.png')
plt.show()

In [ ]:
# finding missing values
print(df.isnull().sum())

In [ ]:
# stalk-root missing vals
for col in df.columns:
    if '?' in df[col].values:
        count = (df[col] == '?').sum()
        print(f"{col}: {count} missing values")

In [ ]:
# Feature distributions

fig, axes = plt.subplots(nrows = 6, ncols = 4, figsize = (20, 20))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    df[col].value_counts().plot(kind = 'bar', ax = axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('')
plt.tight_layout()
plt.show()


some features are pretty imbalanced so we might want to consider removing them

In [ ]:
# chi-squared test
from scipy.stats import chi2_contingency

results = []
for col in df.drop(columns=['target']).columns:
    contingency_table = pd.crosstab(df[col], df['target'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    results.append({'feature': col, 'chi2': chi2, 'p-value': p})
chi2_df = pd.DataFrame(results).sort_values('chi2', ascending=False)
print(chi2_df)

In [ ]:
# visual for chi-squared
plt.figure(figsize=(10, 6))
plt.barh(chi2_df['feature'], chi2_df['chi2'])
plt.xlabel('Chi-Squared')
plt.savefig('../results/chi_squared.png')
plt.show()

the chi-squared results are sorted, so we can see which features have a stronger association with the target var. veil-type has a chi2 of zero, so we should drop. the rest of the vars are statistically significant but their p-values are still pretty low comparatively. should we keep them all or drop a few? not sure!